In [213]:
# build shallow network carbapnemase classifier in pytorch
#logistic regression carbapenemase classifier
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import random
import json
from copy import deepcopy
import warnings
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    confusion_matrix,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# DEVICE = "mps" #mac graphics card
EMB_TYPE = "MeanEmbedding"  # or "CLSEmbedding"
warnings.filterwarnings("ignore")
print("DEVICE:", DEVICE)
print("EMB_TYPE:", EMB_TYPE)

DEVICE: cpu
EMB_TYPE: MeanEmbedding


In [214]:
df = pd.read_csv("embeddings.csv")

class_A = [
    "KPC",
    "IMI",
    "SME",
    #"GES",
    "TEM",
    "SHV",
    "CTX-M",
    "VEB"
]

families = [
    "KPC",
    "NDM",
    "TEM",
    "ADC",
    "CTX-M",
    "PDC",
    "SME",
    "IMI",
    "SHV",
    #"ACT",
    #"GES",
    #"VEB",
    #"OXA",
    "DHA",
    "MOX",
    "CMY",
    "EC",
    #"SME"
    #"FOX"
]
df = df[df["Family"].isin(class_A)]
df["Carbapenemase"].value_counts()

Carbapenemase
0    737
1    261
Name: count, dtype: int64

In [215]:
carb = df[df["Carbapenemase"] == 1]
non_carb = df[df["Carbapenemase"] == 0]
print("carb counts:", carb["Family"].value_counts())
print("non-carb counts:", non_carb["Family"].value_counts())

carb counts: Family
KPC    230
IMI     26
SME      5
Name: count, dtype: int64
non-carb counts: Family
CTX-M    266
SHV      216
TEM      216
VEB       39
Name: count, dtype: int64


In [216]:
#prepare dataset:
X = np.vstack(df[EMB_TYPE].apply(lambda x: np.fromstring(x.strip("[]"), sep=" ")))
y = df["Carbapenemase"].values

#scale features
X_scaled = StandardScaler().fit_transform(X)

In [217]:
# shallow fully connected network
class CarbapenemaseMLP(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(emb_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # returns logits

In [218]:
# training function
def train_one_fold(model, X_train, y_train, epochs=50, lr=1e-4, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    model.train()
    for epoch in range(epochs):
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    return model

In [219]:
# prediction function
def predict_probs(model, X):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()

    return probs

In [220]:
# 5 fold cv
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

metrics = {
    "fold": [],
    "accuracy": [],
    "balanced_accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "roc_auc": [],
    "avg_precision": [],
    "threshold": []
}

best_threshold = 0.5  # replace with your tuned threshold if desired

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Fit scaler on training fold only
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Build model
    model = CarbapenemaseMLP(emb_dim=X_train_scaled.shape[1]) 

    model = train_one_fold(
        model,
        X_train_scaled,
        y_train,
        epochs=50,
        batch_size=32,
    )

    # Predict
    y_prob = predict_probs(model, X_test_scaled)
    y_pred = (y_prob >= best_threshold).astype(int)

    # Metrics
    metrics["fold"].append(fold)
    metrics["accuracy"].append(accuracy_score(y_test, y_pred))
    metrics["balanced_accuracy"].append(balanced_accuracy_score(y_test, y_pred))
    metrics["precision"].append(precision_score(y_test, y_pred, zero_division=0))
    metrics["recall"].append(recall_score(y_test, y_pred, zero_division=0))
    metrics["f1"].append(f1_score(y_test, y_pred, zero_division=0))
    metrics["roc_auc"].append(roc_auc_score(y_test, y_prob))
    metrics["avg_precision"].append(average_precision_score(y_test, y_prob))
    metrics["threshold"].append(best_threshold)

metrics_df = pd.DataFrame(metrics)
print(metrics_df)

   fold  accuracy  balanced_accuracy  precision    recall        f1   roc_auc  \
0     1  0.750000           0.762474   0.512500  0.788462  0.621212  0.843165   
1     2  0.800000           0.814969   0.578947  0.846154  0.687500  0.889293   
2     3  0.785000           0.787383   0.567568  0.792453  0.661417  0.850340   
3     4  0.768844           0.737899   0.546875  0.673077  0.603448  0.809916   
4     5  0.829146           0.803571   0.650000  0.750000  0.696429  0.865385   

   avg_precision  threshold  
0       0.690374        0.5  
1       0.812823        0.5  
2       0.621178        0.5  
3       0.633969        0.5  
4       0.703395        0.5  


In [221]:
# train final model on all data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
final_model = CarbapenemaseMLP(emb_dim=X_scaled.shape[1])
final_model = train_one_fold(
    final_model,
    X_scaled,
    y,
    epochs=50,
    batch_size=32,
)
# save final model and scaler
torch.save(final_model.state_dict(), "final_carbapenemase_mlp_fcn.pth")
with open("scaler_params_fcn.json", "w") as f:
    json.dump({
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist()
    }, f)

#save final model metrics
final_metrics = {
    "accuracy": metrics_df["accuracy"].mean(),
    "balanced_accuracy": metrics_df["balanced_accuracy"].mean(),
    "precision": metrics_df["precision"].mean(),
    "recall": metrics_df["recall"].mean(),
    "f1": metrics_df["f1"].mean(),
    "roc_auc": metrics_df["roc_auc"].mean(),
    "avg_precision": metrics_df["avg_precision"].mean(),
    "threshold": best_threshold
}

#save final metrics to csv
final_metrics_df = pd.DataFrame([final_metrics])
final_metrics_df.to_csv("final_model_metrics_fcn.csv", index=False)
final_metrics_df

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,avg_precision,threshold
0,0.786598,0.781259,0.571178,0.770029,0.654001,0.85162,0.692348,0.5


# problem is non-carbapenemases are getting called as carbapenemases, need to incorporate non class-A carbapenamases and non class A non carbapenemases

In [222]:
# print confusion matrix on all folds


In [223]:
# inference for test sequence
# inference for test sequence
import torch
import json
import numpy as np
from sklearn.preprocessing import StandardScaler

# ── 1. Load scaler params ──────────────────────────────────────────────────────
with open("scaler_params_fcn.json", "r") as f:
    scaler_params = json.load(f)

scaler = StandardScaler()
scaler.mean_ = np.array(scaler_params["mean"])
scaler.scale_ = np.array(scaler_params["scale"])

# ── 2. Load model ──────────────────────────────────────────────────────────────
emb_dim = len(scaler.mean_)
inference_model = CarbapenemaseMLP(emb_dim=emb_dim)
inference_model.load_state_dict(torch.load("final_carbapenemase_mlp_fcn.pth", map_location=DEVICE))
inference_model = inference_model.to(DEVICE)
inference_model.eval()

# ── 3. Prepare test sequence embedding ────────────────────────────────────────
# Replace this with your actual embedding vector (MeanEmbedding from ESM/ProtTrans etc.)
# test_embedding should be a 1D numpy array of shape (emb_dim,)
test_row = df.iloc[0]  # example: first row from the dataframe
test_embedding = np.fromstring(test_row[EMB_TYPE].strip("[]"), sep=" ")

# ── 4. Scale and run inference ─────────────────────────────────────────────────
test_scaled = scaler.transform(test_embedding.reshape(1, -1))
test_tensor = torch.tensor(test_scaled, dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    logit = inference_model(test_tensor)
    probability = torch.sigmoid(logit).item()

print(f"Carbapenemase probability : {probability:.4f}")
print(f"Predicted class           : {'Carbapenemase' if probability >= best_threshold else 'Non-carbapenemase'}")
print(f"True label                : {test_row['Carbapenemase']} (Family: {test_row['Family']})")

Carbapenemase probability : 0.1552
Predicted class           : Non-carbapenemase
True label                : 0 (Family: SHV)


In [224]:
def score_protein(embedding_vector: np.ndarray) -> dict:
    """
    Score a protein sequence embedding using the trained carbapenemase classifier.

    Args:
        embedding_vector: 1D numpy array of shape (emb_dim,) - the protein embedding     
    Returns:
        dict with probability, predicted class, and logit
    """
    # ── 1. Load scaler params ──────────────────────────────────────────────────
    with open("scaler_params_fcn.json", "r") as f:
        scaler_params = json.load(f)

    scaler = StandardScaler()
    scaler.mean_ = np.array(scaler_params["mean"])
    scaler.scale_ = np.array(scaler_params["scale"])

    # ── 2. Load model ──────────────────────────────────────────────────────────
    emb_dim = len(scaler.mean_)
    model = CarbapenemaseMLP(emb_dim=emb_dim)
    model.load_state_dict(torch.load("final_carbapenemase_mlp_fcn.pth", map_location=DEVICE))
    model = model.to(DEVICE)
    model.eval()

    # ── 3. Scale and run inference ─────────────────────────────────────────────
    scaled = scaler.transform(embedding_vector.reshape(1, -1))
    tensor = torch.tensor(scaled, dtype=torch.float32).to(DEVICE)

    with torch.no_grad():
        logit = model(tensor)
        probability = torch.sigmoid(logit).item()

    predicted_class = "Carbapenemase" if probability >= best_threshold else "Non-carbapenemase"

    return {
        "probability": probability,
        "predicted_class": predicted_class,
        "logit": logit.item()
    }


# ── Example usage ──────────────────────────────────────────────────────────────
test_row = df.iloc[0]
test_embedding = np.fromstring(test_row[EMB_TYPE].strip("[]"), sep=" ")

result = score_protein(test_embedding)
print(f"Carbapenemase probability : {result['probability']:.4f}")
print(f"Predicted class           : {result['predicted_class']}")
print(f"True label                : {test_row['Carbapenemase']} (Family: {test_row['Family']})")

Carbapenemase probability : 0.1552
Predicted class           : Non-carbapenemase
True label                : 0 (Family: SHV)
